# AI Power Blackout Predictor — Demo Notebook

This notebook demonstrates the core prediction and analysis capabilities of the AI Power Blackout Predictor platform using the Python SDK.

## Setup
```
pip install blackout-predictor-sdk httpx pandas matplotlib folium
```

In [ ]:
import asyncio
import os

import httpx
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

API_BASE = os.getenv('BLACKOUT_API_URL', 'http://localhost:8000')
API_KEY  = os.getenv('BLACKOUT_API_KEY', '')  # JWT token

HEADERS = {'Authorization': f'Bearer {API_KEY}'} if API_KEY else {}
print(f'Connecting to: {API_BASE}')

## 1. Health Check

In [ ]:
async def check_health():
    async with httpx.AsyncClient(base_url=API_BASE, timeout=10) as client:
        r = await client.get('/health')
        return r.json()

health = await check_health()
print('Health:', health)

## 2. Fetch Prediction Heatmap

In [ ]:
async def get_heatmap():
    async with httpx.AsyncClient(base_url=API_BASE, headers=HEADERS, timeout=30) as client:
        r = await client.get('/api/v1/predictions/heatmap')
        r.raise_for_status()
        return r.json()

heatmap = await get_heatmap()
features = heatmap.get('features', [])
print(f'Total cells: {len(features)}')

## 3. Analyze Risk Distribution

In [ ]:
rows = []
for f in features:
    props = f['properties']
    coords = f['geometry']['coordinates']
    rows.append({
        'h3_index': props['h3_index'],
        'probability': props['probability'],
        'risk_level': props['risk_level'],
        'lng': coords[0],
        'lat': coords[1],
    })

df = pd.DataFrame(rows)
print(df.describe())
df.head()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Risk level distribution
risk_counts = df['risk_level'].value_counts()
colors = {'low': '#22c55e', 'medium': '#f59e0b', 'high': '#ef4444', 'critical': '#7c3aed'}
risk_counts.plot(
    kind='bar', ax=axes[0],
    color=[colors.get(r, '#94a3b8') for r in risk_counts.index]
)
axes[0].set_title('Risk Level Distribution')
axes[0].set_xlabel('Risk Level')
axes[0].set_ylabel('Number of Cells')
axes[0].tick_params(axis='x', rotation=0)

# Probability histogram
df['probability'].plot(kind='hist', bins=20, ax=axes[1], color='#6366f1', edgecolor='white')
axes[1].set_title('Outage Probability Distribution')
axes[1].set_xlabel('Probability')
axes[1].set_ylabel('Frequency')

plt.tight_layout()
plt.show()

## 4. Interactive Map (Folium)

In [ ]:
try:
    import folium
    import h3

    center_lat = df['lat'].mean() if len(df) else 0
    center_lng = df['lng'].mean() if len(df) else 0
    m = folium.Map(location=[center_lat, center_lng], zoom_start=9, tiles='CartoDB dark_matter')

    for _, row in df.iterrows():
        color = colors.get(row['risk_level'], '#94a3b8')
        folium.CircleMarker(
            location=[row['lat'], row['lng']],
            radius=8,
            color=color,
            fill=True,
            fill_opacity=0.7,
            popup=f"{row['h3_index']}<br>P={row['probability']:.1%}<br>{row['risk_level']}",
        ).add_to(m)

    display(m)
except ImportError:
    print('Install folium and h3 for map: pip install folium h3')

## 5. SHAP-style Feature Explanation for a Single Cell

In [ ]:
async def explain_cell(h3_index: str):
    async with httpx.AsyncClient(base_url=API_BASE, headers=HEADERS, timeout=15) as client:
        r = await client.get(f'/api/v1/predictions/cell/{h3_index}/explain')
        r.raise_for_status()
        return r.json()

if len(df) > 0:
    top_cell = df.sort_values('probability', ascending=False).iloc[0]['h3_index']
    explanation = await explain_cell(top_cell)
    print(f'Top risk cell: {top_cell}')
    print(f"Probability: {explanation.get('probability', 0):.1%}")
    print(f"Top factor: {explanation.get('top_factor')}")

    weights = explanation.get('feature_weights', {})
    if weights:
        fig, ax = plt.subplots(figsize=(8, 4))
        sorted_items = sorted(weights.items(), key=lambda x: x[1], reverse=True)
        labels, values = zip(*sorted_items)
        bar_colors = ['#ef4444' if v > 0.5 else '#f59e0b' if v > 0.3 else '#22c55e' for v in values]
        ax.barh(labels, values, color=bar_colors)
        ax.set_xlabel('Feature Weight')
        ax.set_title(f'Risk Factors — {top_cell}')
        plt.tight_layout()
        plt.show()
else:
    print('No prediction data available')

## 6. Carbon Impact Estimate

In [ ]:
async def carbon_impact(hours=2.0, load_mw=0.5):
    async with httpx.AsyncClient(base_url=API_BASE, headers=HEADERS, timeout=10) as client:
        r = await client.get(f'/api/v1/carbon/impact?hours={hours}&load_mw={load_mw}')
        r.raise_for_status()
        return r.json()

impact = await carbon_impact()
print(f"Affected cells:      {impact.get('affected_cells')}")
print(f"kWh lost:            {impact.get('kwh_lost'):.1f}")
print(f"CO2 avoided (kg):    {impact.get('co2_kg_avoided'):.1f}")
print(f"CO2 avoided (t):     {impact.get('co2_tonnes_avoided'):.4f}")

## 7. AI Assistant (RAG)

In [ ]:
async def ask_assistant(question: str, h3_index: str | None = None):
    async with httpx.AsyncClient(base_url=API_BASE, headers=HEADERS, timeout=20) as client:
        r = await client.post('/api/v1/assistant/ask', json={'question': question, 'h3_index': h3_index})
        r.raise_for_status()
        return r.json()

answer = await ask_assistant('What areas have the highest outage risk right now?')
print('Assistant:', answer.get('answer'))